In [1]:
!pip install requests beautifulsoup4 python-dateutil


[notice] A new release of pip is available: 25.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import os
import re
import time
import requests
from datetime import datetime
from bs4 import BeautifulSoup
from dateutil.relativedelta import relativedelta

# ------------------------------------------------------------------------------
# CONFIGURATION & CONSTANTS
# ------------------------------------------------------------------------------
START_DATE = datetime(2022, 1, 1)
END_DATE = datetime(2026, 9, 30)
OUTPUT_DIR = "scraped_incubators_html"
USER_AGENT = "Mozilla/5.0 (Windows NT 10.0; Win64; x64) StartupResearchBot/1.0"

# Target Top 20 Incubators (Methodology A) with Homepage and About Us URLs
INCUBATORS = [
    {"name": "NASSCOM_10000_Startups", 
        "homepage": "https://nasscom.in", 
        "about": "https://nasscom.in/deeptech/deeptechclub/startups-listing.php"},
    {"name": "Kerala_Startup_Mission", "homepage": "https://startupmission.kerala.gov.in", "about": "https://startupmission.kerala.gov.in/about"},
    {"name": "T_Hub", "homepage": "https://t-hub.co", "about": "https://t-hub.co/about-us"},
    {"name": "IIMB_NSRCEL", "homepage": "https://nsrcel.org", "about": "https://nsrcel.org/about-us"},
    {"name": "i_Hub_Gujarat", "homepage": "http://ihubgujarat.in", "about": "http://ihubgujarat.in/about"},
    {"name": "Amity_Innovation_Incubator", "homepage": "https://www.amity.edu/aiit", "about": "https://www.amity.edu/aiit/about.asp"},
    {"name": "IIMA_CIIE_CO", "homepage": "https://ciie.co", "about": "https://ciie.co/about"},
    {"name": "C_CAMP", "homepage": "https://www.ccamp.res.in", "about": "https://www.ccamp.res.in/about-us"},
    {"name": "CSIR_NCL_Venture_Center", "homepage": "https://www.venturecenter.co.in", "about": "https://www.venturecenter.co.in/about.php"},
    {"name": "IIT_Madras_IITMIC", "homepage": "https://incubation.iitm.ac.in", "about": "https://incubation.iitm.ac.in/about-us"},
    {"name": "KIIT_TBI", "homepage": "https://kiitincubator.in", "about": "https://kiitincubator.in/about-us"},
    {"name": "Social_Alpha", "homepage": "https://www.socialalpha.org", "about": "https://www.socialalpha.org/about-us"},
    {"name": "AIC_Banasthali", "homepage": "https://www.banasthali.org/aic", "about": "https://www.banasthali.org/aic/about"},
    {"name": "IIT_Delhi_FITT", "homepage": "https://fitt-iitd.in", "about": "https://fitt-iitd.in/about-us"},
    {"name": "IIT_Bombay_SINE", "homepage": "https://sineiitb.org", "about": "https://sineiitb.org/sine/about"},
    {"name": "IIM_Calcutta_IIMCIP", "homepage": "https://www.iimcip.org", "about": "https://www.iimcip.org/about-us"},
    {"name": "IIT_Kanpur_SIIC", "homepage": "https://siicincubator.com", "about": "https://siicincubator.com/about-us"},
    {"name": "Villgro_Innovations", "homepage": "https://www.villgro.org", "about": "https://www.villgro.org/about-us"},
    {"name": "JSSATE_STEP", "homepage": "https://jssstep.in", "about": "https://jssstep.in/about"},
    {"name": "IIT_Hyderabad_i_TIC", "homepage": "https://itic.iith.ac.in", "about": "https://itic.iith.ac.in/about"}
]

# ------------------------------------------------------------------------------
# HELPER FUNCTIONS
# ------------------------------------------------------------------------------
def generate_monthly_checkpoints(start_dt, end_dt):
    """Generates tuples of (Year, Month, YearMonthString YYYYMM) from start_dt to end_dt."""
    current = start_dt
    checkpoints = []
    while current <= end_dt:
        checkpoints.append((current.year, current.month, current.strftime("%Y%m")))
        current += relativedelta(months=1)
    return checkpoints

def sanitize_filename(name):
    """Cleans up strings to make valid OS filenames."""
    return re.sub(r'[^a-zA-Z0-9_\-]', '_', name)

def get_earliest_wayback_snapshot(target_url, yyyymm):
    """Queries Wayback CDX API with automatic retry and backoff on timeout."""
    cdx_api_url = "http://web.archive.org/cdx/search/cdx"
    params = {
        "url": target_url,
        "output": "json",
        "from": f"{yyyymm}01000000",
        "to": f"{yyyymm}31235959",
        "filter": "statuscode:200",
        "limit": "10"
    }
    headers = {"User-Agent": USER_AGENT}
    
    # Try up to 4 times with increasing wait times if Wayback times out
    for attempt in range(4):
        try:
            response = requests.get(cdx_api_url, params=params, headers=headers, timeout=25)
            if response.status_code == 200:
                data = response.json()
                if len(data) > 1:
                    timestamp = data[1][1]
                    original_url = data[1][2]
                    return f"https://web.archive.org/web/{timestamp}/{original_url}", timestamp
            return None, None
        except requests.exceptions.RequestException:
            if attempt == 3:
                # Silently skip if all 4 attempts fail so the script doesn't crash
                return None, None
            time.sleep(3 * (attempt + 1))  # Wait longer between each retry (3s, 6s, 9s)
    return None, None

def fetch_and_save_html(url, filepath):
    """Fetches HTML content from live or Wayback Machine URL and saves to disk."""
    headers = {"User-Agent": USER_AGENT}
    try:
        response = requests.get(url, headers=headers, timeout=25)
        if response.status_code == 200:
            with open(filepath, "w", encoding="utf-8") as f:
                f.write(response.text)
            return True
    except Exception as e:
        print(f"   [Download Error] Failed to fetch {url}: {e}")
    return False

# ------------------------------------------------------------------------------
# MAIN EXECUTION ROUTINE
# ------------------------------------------------------------------------------
def main():
    print("=================================================================")
    print("  AUTOMATED INCUBATOR WEBSITE & ARCHIVE SCRAPER (2022 - 2026)  ")
    print("=================================================================\n")
    
    os.makedirs(OUTPUT_DIR, exist_ok=True)
    monthly_checkpoints = generate_monthly_checkpoints(START_DATE, END_DATE)
    print(f"Total Monthly Checkpoints: {len(monthly_checkpoints)} (Jan 2022 to Sep 2026)")
    print(f"Total Incubators: {len(INCUBATORS)}")
    print(f"Target Pages per Incubator: 2 (Homepage & About Us)")
    print(f"Output Directory: ./{OUTPUT_DIR}/\n")
    
    total_files_downloaded = 0

    for idx, inc in enumerate(INCUBATORS, 1):
        inc_name = sanitize_filename(inc["name"])
        print(f"\n[{idx}/{len(INCUBATORS)}] Processing Incubator: {inc_name}")
        
        # ----------------------------------------------------------------------
        # 1. FETCH CURRENT (LIVE) PAGES
        # ----------------------------------------------------------------------
        print("  --> Fetching CURRENT (Live) Pages...")
        live_dir = os.path.join(OUTPUT_DIR, inc_name, "CURRENT_LIVE")
        os.makedirs(live_dir, exist_ok=True)
        
        # Live Homepage
        hp_path = os.path.join(live_dir, "homepage_current.html")
        if fetch_and_save_html(inc["homepage"], hp_path):
            total_files_downloaded += 1
            
        # Live About Us
        ab_path = os.path.join(live_dir, "about_us_current.html")
        if fetch_and_save_html(inc["about"], ab_path):
            total_files_downloaded += 1
            
        time.sleep(1)  # Polite delay
        
        # ----------------------------------------------------------------------
        # 2. FETCH HISTORICAL ARCHIVED PAGES (WAYBACK MACHINE: JAN 2022 - SEP 2026)
        # ----------------------------------------------------------------------
        print("  --> Fetching ARCHIVED Pages (Wayback Machine 2022-2026)...")
        arch_dir = os.path.join(OUTPUT_DIR, inc_name, "ARCHIVED_MONTHLY")
        os.makedirs(arch_dir, exist_ok=True)
        
        for year, month, yyyymm in monthly_checkpoints:
            # A. Process Homepage Archive
            hp_arch_file = os.path.join(arch_dir, f"{yyyymm}_homepage.html")
            if not os.path.exists(hp_arch_file):
                wb_url, ts = get_earliest_wayback_snapshot(inc["homepage"], yyyymm)
                if wb_url and fetch_and_save_html(wb_url, hp_arch_file):
                    total_files_downloaded += 1
                time.sleep(0.5)  # Throttle Wayback requests
            
            # B. Process About Us Archive
            ab_arch_file = os.path.join(arch_dir, f"{yyyymm}_about_us.html")
            if not os.path.exists(ab_arch_file):
                wb_url, ts = get_earliest_wayback_snapshot(inc["about"], yyyymm)
                if wb_url and fetch_and_save_html(wb_url, ab_arch_file):
                    total_files_downloaded += 1
                time.sleep(0.5)  # Throttle Wayback requests

    print("\n=================================================================")
    print(f" SCRAPING COMPLETE! Total HTML files saved: {total_files_downloaded}")
    print("=================================================================")

if __name__ == "__main__":
    main() 

  AUTOMATED INCUBATOR WEBSITE & ARCHIVE SCRAPER (2022 - 2026)  

Total Monthly Checkpoints: 57 (Jan 2022 to Sep 2026)
Total Incubators: 20
Target Pages per Incubator: 2 (Homepage & About Us)
Output Directory: ./scraped_incubators_html/


[1/20] Processing Incubator: NASSCOM_10000_Startups
  --> Fetching CURRENT (Live) Pages...
  --> Fetching ARCHIVED Pages (Wayback Machine 2022-2026)...

[2/20] Processing Incubator: Kerala_Startup_Mission
  --> Fetching CURRENT (Live) Pages...
  --> Fetching ARCHIVED Pages (Wayback Machine 2022-2026)...

[3/20] Processing Incubator: T_Hub
  --> Fetching CURRENT (Live) Pages...
  --> Fetching ARCHIVED Pages (Wayback Machine 2022-2026)...
   [Download Error] Failed to fetch https://web.archive.org/web/20260702010830/https://www.t-hub.co/: HTTPSConnectionPool(host='web.archive.org', port=443): Max retries exceeded with url: /web/20260702010830/https://www.t-hub.co/ (Caused by ConnectTimeoutError(<urllib3.connection.HTTPSConnection object at 0x000001FAD40D